In [1]:
from pathlib import Path
from collections import Counter
import statistics
import random

## 1. Imports

We import the Python tools needed for dataset exploration.

- `Path` is used to work with file and directory paths.
- `Counter` is used to count how frequently characters occur.
- `statistics` provides statistical functions such as the median.
- `random` is used later to inspect reproducible random examples.

## 2. Dataset location and configuration

The dataset is provided on the GU GPU server.

The dataset contains three languages:
- English (`en`)
- Turkish (`tr`)
- Mandarin Chinese (`zh`)

It is divided into:
- training data
- validation data
- test data

The training data is used for learning tokenizer vocabularies and training models.
The validation data is used for development and experimental decisions.
The test data should only be used for final evaluation.

We define the dataset location, languages, and splits once so that the rest of the notebook can
reuse these values instead of repeatedly typing file paths.

In [2]:
DATA_DIR = Path("/srv/data/lt2326-h26/a1")
LANGUAGES = ["en", "tr", "zh"]
SPLITS = ["train", "valid", "test"]


## 3. Check that the dataset path exists

Before doing any analysis, we verify that the path is valid.

`Path.exists()` returns `True` if the specified file or directory exists.

This is a simple but useful debugging step: it lets us detect an incorrect path before running
the rest of the notebook.

In [3]:
print(DATA_DIR)
print(DATA_DIR.exists())


/srv/data/lt2326-h26/a1
True


## 4. Inspect the dataset structure

The dataset contains separate files for each language and split.

We recursively search through the dataset directory to see which files are available.

This is useful because dataset exploration should begin by understanding the physical organization
of the data before loading it.

In [4]:
for path in DATA_DIR.rglob("*"):
    if path.is_file():
        print(path)

/srv/data/lt2326-h26/a1/README.txt
/srv/data/lt2326-h26/a1/metadata.json
/srv/data/lt2326-h26/a1/test/en.txt
/srv/data/lt2326-h26/a1/test/tr.txt
/srv/data/lt2326-h26/a1/test/zh.txt
/srv/data/lt2326-h26/a1/tokenizer/balanced.txt
/srv/data/lt2326-h26/a1/tokenizer/imbalanced_10-1-1.txt
/srv/data/lt2326-h26/a1/train/en.txt
/srv/data/lt2326-h26/a1/train/tr.txt
/srv/data/lt2326-h26/a1/train/zh.txt
/srv/data/lt2326-h26/a1/valid/en.txt
/srv/data/lt2326-h26/a1/valid/tr.txt
/srv/data/lt2326-h26/a1/valid/zh.txt


## 5. Loading sentences

Each dataset file contains one sentence per line.

The `load_sentences()` function:
1. constructs the path for a particular split and language,
2. reads the file using UTF-8 encoding,
3. separates the file into individual lines.

`splitlines()` is used instead of manually splitting on `"\n"` because it handles different line-ending
conventions more generally.

The result is a Python list where each element represents one sentence.

In [5]:
def load_sentences(split, language):
    path = DATA_DIR / split / f"{language}.txt"
    text = path.read_text(encoding="utf-8")
    return text.splitlines()

## 6. Test the sentence-loading function

Before using the function for the whole dataset, we test it on the English training data.

We inspect:
- the returned object type,
- the number of sentences,
- the first three sentences.

Printing only the first few sentences prevents the notebook from producing a huge output.

In [6]:
english_train = load_sentences("train", "en")
print(type(english_train))
print(len(english_train))
print(english_train[:3])

<class 'list'>
36004
['Since Jack Kirby had more than enough assignments, Lee assigned John Buscema the task of illustrating the new book."', 'Inuit means more than one, one person is an "Inuk".', 'Sunday edition until his death in 2017.']


## 7. Load the complete dataset into memory

We create a nested dictionary containing all languages and splits.

The resulting structure is:

data
├── train
│   ├── en
│   ├── tr
│   └── zh
├── valid
│   ├── en
│   ├── tr
│   └── zh
└── test
    ├── en
    ├── tr
    └── zh

For example:

`data["train"]["en"]` gives all English training sentences.

`data["valid"]["zh"]` gives all Chinese validation sentences.

This structure makes it easier to perform the same analysis across languages and splits.

In [7]:
data = {}
for split in SPLITS:
    data[split] = {}
    for language in LANGUAGES:
        data[split][language] = load_sentences(split, language)

In [8]:
data["train"]["en"][:3]

['Since Jack Kirby had more than enough assignments, Lee assigned John Buscema the task of illustrating the new book."',
 'Inuit means more than one, one person is an "Inuk".',
 'Sunday edition until his death in 2017.']

In [9]:
data["train"]["tr"][:3]

['Ekonomi alanındaki bu gelişmelerin yanında, okuma-yazma bilmezliğe karşı yoğun bir savaşım görüldü.',
 "Babasının isteği üzerine 1864'te hukukçu olmak için memur olarak gittiği Siyalkut'ta başarısız olunca 1868'de Kadıyan'a dönerek inzivaya çekildi.",
 "'nde muhafaza edilmesine karar verilmiştir."]

In [10]:
data["train"]["zh"][:3]

['他直到死前都沒有親自管理過公國。',
 '，面積205平方公里，海拔高度611米，2007年人口6,597，人口密度每平方公里32人。',
 '帝谓：“善长虽无汗马劳，然事朕久，给军食，功甚大，宜进封大国。”']

In [11]:
data["valid"]["en"][:3]


['She played in two matches at this championship.',
 'After these victories, he was appointed as the organizing secretary of DMK in southern Tamil Nadu districts.',
 "In 1536, the Act of Union was passed under Henry's rule which had a long-lasting effect on Wales as a nation."]

In [12]:
data["valid"]["tr"][:3]

["Germantown akademisine devam ettikten sonra 1869-1870 yılları Fransa ve Almanya'da okullarda geçti.",
 'Arşivde yer alan farklı iki parça da 2017\'de "3 Gerald Remix / 24 TSIM 2" ismiyle Michigan\'da bir plak mağazasında satılmıştır.',
 'Kayseri iline 54 km, Bünyan ilçesine 19 km uzaklıktadır.']

In [13]:
data["valid"]["zh"][:3]

['，也是一所专门提供研究生及以上级别课程的私立精英级大学。',
 '藝術家會在作品中控制灰分的百分比，若降低灰的比例，控制的程度更高 Ford & Impey, 46-50 。',
 '這不是外來的概念，傳統是由瑪泰在死亡前以一種被稱為「馬卡加」(mavaega)的意志形式出現，後來則由公共信託辦公室和法律從業者負責處理遺產管理。']

In [14]:
data["test"]["en"][:3]

['Siegfried kills Mime by chopping his head off with the sword.',
 "Oklahoma City is the state's capital and largest city by population and land area.",
 'The troubles stemmed from the cultural clash between the tribal groups and the plains people.']

In [15]:
data["test"]["tr"][:3]


['Rivayete göre kırklar mevkiinden geçen gelin buradaki kavak ağaçlarının Tanrıya secde ettiklerini görür.',
 'Bundan dolayı alternatif diyotlar verimli güç dönüşümü için kullanılmak zorundadırlar.',
 'Dünya Gümrük Örgütü bünyesinde imzalanan bazı uluslararası sözleşmelere Türkiye de taraf olmuştur.']

In [16]:
data["test"]["zh"][:3]

['广场公园共分為始绿园、春之园、感觉园、嗅觉园、地质园、触觉园、干河园、芳草园、视觉园、听觉园、味觉园、自然生态园、音乐园、大树主题广场、市民林等区域。',
 '，并获封三等承恩公，此宅遂升格为承恩公府。',
 '，“玛夏（Masher）”则有可能来源于mashadar（意为前镗枪）或者masha（意为女王）。']

## 8. Number of sentences

We count the number of sentences in every language and split.

This is useful for understanding how the dataset is distributed.

An important point is that the languages do not contain the same number of sentences.
This does not necessarily mean that one language has more underlying text, because the dataset
was balanced approximately by Unicode character count rather than sentence count.

In [17]:
for split in SPLITS:
    print(f"\n {"=" * 10} {split.upper()} {"=" * 10}")
    for language in LANGUAGES:
        sentences = data[split][language]
        print(language, len(sentences))


 ========== TRAIN ==========
en 36004
tr 30379
zh 75962

 ========== VALID ==========
en 4540
tr 3735
zh 9554

 ========== TEST ==========
en 4472
tr 3792
zh 9430


## 9. Total Unicode characters

We calculate the total number of characters in each dataset.

Python strings use Unicode, so `len(sentence)` counts Unicode characters rather than UTF-8 bytes.

This distinction is important because the dataset was balanced using Unicode character counts.

For example, a Chinese character may require multiple bytes when stored as UTF-8, but it is still
one Unicode character and therefore one character-level token in our character tokenizer.

In [18]:
#calculating total characters
def total_characters(sentences):
    return sum(len(sentence) for sentence in sentences)

In [19]:
english_train = data["train"]["en"]
print("Total characters in English train:", total_characters(english_train))

english_valid = data["valid"]["en"]
print("Total characters in English valid:", total_characters(english_valid))

Total characters in English train: 2946215
Total characters in English valid: 368269


## 10. Dataset size and average sentence length

For every language and split we calculate:
- number of sentences
- total number of Unicode characters
- average number of characters per sentence

The average is calculated as:

total characters / number of sentences

For the character tokenizer, one character corresponds to one token, so the average number of
characters per sentence is also the average number of character tokens per sentence.

In [20]:
#calculating statistics for every split and language
for split in SPLITS:
    print(f"\n {"=" * 10} {split.upper()} {"=" * 10}")
    for language in LANGUAGES:
        sentences = data[split][language]
        n_sentences = len(sentences)
        n_characters = total_characters(sentences)
        average_length = n_characters / n_sentences

        print(language, "sentences =", n_sentences,
        "characters =", n_characters,
        "avg chars/sentence =", round(average_length, 2))



 ========== TRAIN ==========
en sentences = 36004 characters = 2946215 avg chars/sentence = 81.83
tr sentences = 30379 characters = 2946140 avg chars/sentence = 96.98
zh sentences = 75962 characters = 2946177 avg chars/sentence = 38.78

 ========== VALID ==========
en sentences = 4540 characters = 368269 avg chars/sentence = 81.12
tr sentences = 3735 characters = 368383 avg chars/sentence = 98.63
zh sentences = 9554 characters = 368266 avg chars/sentence = 38.55

 ========== TEST ==========
en sentences = 4472 characters = 368309 avg chars/sentence = 82.36
tr sentences = 3792 characters = 368342 avg chars/sentence = 97.14
zh sentences = 9430 characters = 368265 avg chars/sentence = 39.05


## 11. Sentence-length distribution

An average sentence length does not tell us how sentence lengths are distributed.

For example, a dataset could contain mostly short sentences and a few extremely long sentences.
The average alone would hide this variation.

Therefore, we collect the length of every sentence for each language.

In [21]:
sentence_lengths = {}
for language in LANGUAGES:
    sentences = data["train"][language]

    sentence_lengths[language] = [len(sentence) for sentence in sentences]

In [22]:
sentence_lengths["en"][:10]

[116, 51, 39, 223, 49, 30, 87, 45, 92, 40]

## 12. Sentence-length statistics

We calculate:
- minimum sentence length
- maximum sentence length
- average sentence length
- median sentence length

The median is the middle value after sorting the sentence lengths.

The median is useful because it is less affected by unusually long or short sentences than the mean.

In [23]:
for language in LANGUAGES:
    lengths = sentence_lengths[language]

    print(f"\n {"=" * 10} {language.upper()} {"=" * 10}")
    print("Minimum:", min(lengths))
    print("Maximum:", max(lengths))
    print("Average:", round(sum(lengths) / len(lengths), 2))
    print("Median:", statistics.median(lengths))


 ========== EN ==========
Minimum: 15
Maximum: 255
Average: 81.83
Median: 72.0

 ========== TR ==========
Minimum: 15
Maximum: 255
Average: 96.98
Median: 88

 ========== ZH ==========
Minimum: 12
Maximum: 255
Average: 38.78
Median: 31.0


## 13. Inspect shortest and longest sentences

Statistics tell us how large or small sentences are, but actual examples help us understand
what those statistics represent.

We therefore inspect the shortest and longest sentence in each language.

This can also reveal unusual data examples, such as very short fragments, titles, numbers,
or unusually long Wikipedia sentences.

In [24]:
for language in LANGUAGES:
    sentences = data["train"][language]

    shortest_sentence = min(sentences, key=len)
    longest_sentence = max(sentences, key=len)

    print(f"\n {"=" * 10} {language.upper()} {"=" * 10}")
    print("Shortest sentence:", shortest_sentence)
    print("Length:", len(shortest_sentence))
    print("Longest sentence:", longest_sentence)
    print("Length:", len(longest_sentence))


 ========== EN ==========
Shortest sentence: Adams is a top-
Length: 15
Longest sentence: Before he moved to Vermont in August 2002, he was director of the University of Maryland, College Park Institute for Ecological Economics, and a professor in the Center for Environmental Science, at Solomons, and in the Biology Department at College Park.
Length: 255

 ========== TR ==========
Shortest sentence: 'a teslim eder.
Length: 15
Longest sentence: Sonuç olarak, bu durumda polimer parçacıklarının ya sıfır radikali (hareketsiz hali) ya da 1 radikali (polimer büyüme hali) ya da parçacık hesabı başına serbest radikaller için göz ardı edilebilen çok kısa bir 2 radikal (sonlandırma hali) periyodu vardır.
Length: 255

 ========== ZH ==========
Shortest sentence: 連坐：接坐，指有罪同坐。
Length: 12
Longest sentence: ，一群学生慷慨激昂地宣誓：要用生命和热血保卫毛主席、保卫党中央。1966年8月《人民日报》转载清华大学附属中学红卫兵大字报《无产阶级的革命造反精神万岁》、《再论无产阶级的革命造反精神万岁》、《三论无产阶级的革命造反精神万岁》、《打碎旧世界，创立新世界——无产阶级革命造反精神万岁》，宣扬《红卫兵小将的革命造反精神好得很》、《北京城里尽朝辉》 1966年9月28日人民日报《北京城里尽朝辉》

## 14. Inspect random examples

Looking only at the first few sentences can give a biased impression of the dataset.

We therefore inspect random examples from each language.

A fixed random seed is used so that the same examples are selected every time the notebook runs.
This makes the exploration reproducible.

In [25]:
random.seed(2326)

In [26]:
for language in LANGUAGES:
    sentences = data["train"][language]

    print(f"\n===== {language.upper()} =====")

    for sentence in random.sample(sentences, 3):
        print(sentence)


===== EN =====
He won as the underdog in the 1968 presidential election, winning 65.6% of the vote on a 92.2% voter turnout running against Gunnar Thoroddsen.
It is unknown what happened to her severed head.
On 19 December 2011, Stojaković announced that he was retiring from playing professional basketball.

===== TR =====
Jennifer Leann Carpenter (d. 7 Aralık 1979) Amerikalı oyuncudur.
Zeki Yavru 1991 yılında Trabzon'da doğdu.
Kendi sanatsal kişiliği hakkında konuşurken "Herhangi bir müzik tarzıyla kısıtlamıyorum kendimi.

===== ZH =====
国防警备官校”也在原军事英语学校的校址上成立，开始开展更为深入的美式军事教学与训练。
在未找到射击目标之后，卢刚又进入了行政大楼，
文景二帝都是黄老思想的推崇者，主张无为而治，休养生息。


## 15. Character vocabulary size

We now examine the character vocabulary of each language.

A character vocabulary is the set of distinct characters that occur in the training data.

For example:

"hello"

contains four unique characters:

h, e, l, o

because `l` occurs twice.

This is different from the total number of character occurrences.

The character tokenizer uses the union of the characters observed across all three training languages.

In [27]:
#character vocabulary size
for language in LANGUAGES:
    sentences = data["train"][language]
    all_text = "".join(sentences)
    vocabulary = set(all_text)
    print(language, "unique characters =", len(vocabulary))
    

en unique characters = 678
tr unique characters = 508
zh unique characters = 9215


## 16. Characters shared across languages

A multilingual tokenizer has to represent multiple languages using one vocabulary.

We therefore investigate which characters occur in multiple languages.

A Python set intersection (`&`) returns elements that occur in both sets.

The intersection of all three language vocabularies tells us which characters occur in English,
Turkish, and Chinese training data.

In [28]:
#finding shared characters
char_sets = {}

for language in LANGUAGES:
    sentences = data["train"][language]
    text = "".join(sentences)

    char_sets[language] = set(text)

In [29]:
shared_all = (
    char_sets["en"]
    & char_sets["tr"]
    & char_sets["zh"]
)

print("Characters shared by all three:", len(shared_all))

Characters shared by all three: 262


## 17. Language-specific characters

We can also identify characters that occur in one language but not in the other two.

Set difference (`-`) removes elements that occur in another set.

This helps us understand which parts of the character vocabulary are shared and which parts
are specific to individual languages.

In [30]:
#finding language specific characters

en_only = (
    char_sets["en"]
    - char_sets["tr"]
    - char_sets["zh"]
)

tr_only = (
    char_sets["tr"]
    - char_sets["en"]
    - char_sets["zh"]
)

zh_only = (
    char_sets["zh"]
    - char_sets["en"]
    - char_sets["tr"]
)

In [31]:
print("EN-only:", len(en_only))
print("TR-only:", len(tr_only))
print("ZH-only:", len(zh_only))

EN-only: 91
TR-only: 104
ZH-only: 8550


In [32]:
print("EN-only characters:", sorted(en_only))
print("TR-only characters:", sorted(tr_only))
#print("ZH-only characters:", sorted(zh_only))

EN-only characters: ['¡', '¢', 'ą', 'Ć', 'Ľ', 'ľ', 'ņ', 'ň', 'ŷ', 'Ż', 'ż', 'ǎ', 'ǚ', 'ʊ', '̈', 'Θ', 'Е', 'Й', 'כ', 'מ', 'ث', 'ش', 'আ', 'দ', 'ন', 'ফ', 'ব', 'ম', 'য', 'র', 'হ', 'া', 'ি', 'ে', '্', 'ท', 'ป', 'ย', 'ร', 'ษ', 'ำ', 'ไ', 'ᑐ', 'ᑕ', 'ᒡ', 'ᗸ', 'ᘅ', 'ḍ', 'Ḥ', 'ḷ', 'ṃ', 'ṟ', 'Ṣ', 'ạ', 'ế', 'ễ', 'ồ', 'ở', 'ủ', 'ỹ', '‑', '\u202a', '\u202c', '№', '⅕', '∃', '∪', 'ㅂ', 'ㅅ', 'ㅇ', '囯', '釈', '간', '고', '나', '니', '라', '멈', '빛', '빠', '언', '저', '종', '천', '춰', '침', '탱', '하', '형', '훗', 'ﬁ']
TR-only characters: ['¼', 'Í', 'Ú', 'ĉ', 'ė', 'Ğ', 'Ġ', 'ĥ', 'į', 'ĵ', 'ŝ', 'Ǝ', 'ƴ', 'ǧ', 'Ș', 'ț', 'ɲ', 'ʾ', '̂', '͵', 'Έ', 'Α', 'Η', 'Κ', 'Τ', 'έ', 'χ', 'ώ', 'Ц', 'ј', 'ќ', 'ң', 'ү', 'Ա', 'ա', 'գ', 'յ', 'պ', 'ָ', 'ּ', 'ג', 'ח', 'ץ', 'آ', 'َ', 'ُ', 'ِ', 'پ', 'ა', 'ბ', 'გ', 'დ', 'ე', 'ვ', 'ზ', 'თ', 'ი', 'კ', 'ლ', 'მ', 'ნ', 'ო', 'რ', 'ს', 'ტ', 'უ', 'ქ', 'შ', 'ც', 'ძ', 'წ', 'ხ', 'ầ', 'ứ', 'ử', 'ἔ', 'ῖ', '‚', '‟', '₺', 'ℓ', '↨', '∈', '∉', '≠', '각', '과', '국', '는', '디', '먹', '변', '송', '연', '온', '외', '정', '차', '책'

## 18. Character frequency

Vocabulary size tells us which characters exist, but it does not tell us how frequently they occur.

We use `Counter` to count every character occurrence.

The most frequent characters are useful for understanding the statistical structure of the corpus.

This becomes particularly relevant for BPE because BPE learns reusable units based on patterns
and frequencies in the training data.

In [33]:
#character frequency analysis
for language in LANGUAGES:
    sentences = data["train"][language]
    text = "".join(sentences)

    counter = Counter(text)
    print(f"\n===== {language.upper()} =====")
    print(counter.most_common(20))


===== EN =====
[(' ', 474476), ('e', 287293), ('a', 194784), ('t', 189999), ('o', 166130), ('n', 160892), ('i', 160768), ('s', 150819), ('r', 146376), ('h', 114132), ('l', 94270), ('d', 84727), ('c', 65481), ('u', 56674), ('m', 56298), ('f', 46422), ('p', 43171), ('g', 40940), ('w', 38803), ('.', 38152)]

===== TR =====
[(' ', 360375), ('a', 283915), ('e', 222643), ('i', 207692), ('n', 185010), ('r', 177914), ('l', 171562), ('ı', 117691), ('d', 104628), ('k', 103707), ('t', 90199), ('m', 77401), ('s', 76502), ('u', 73692), ('y', 67773), ('o', 63957), ('b', 45149), ('ü', 42514), ('ş', 37619), ('.', 32432)]

===== ZH =====
[('，', 123185), ('的', 70916), ('。', 60726), (' ', 45169), ('1', 32732), ('i', 28357), ('0', 25678), ('t', 23794), ('、', 23167), ('a', 22732), ('在', 22612), ('年', 22266), ('2', 21346), ('一', 21134), ('n', 20461), ('o', 18041), ('人', 17891), ('有', 16864), ('中', 16847), ('是', 16714)]


## 19. Whitespace analysis

We investigate how frequently spaces occur in each language.

English and Turkish normally use spaces to separate words, while Mandarin Chinese generally does
not use spaces between every word.

This difference is important for multilingual tokenization because a tokenizer trained on all
three languages observes very different character sequences.

In [34]:
#investigating whitespaces

for language in LANGUAGES:
    sentences = data["train"][language]
    text = "".join(sentences)

    spaces = text.count(" ")
    total = len(text)

    print(
        language,
        "spaces =", spaces,
        "space ratio =", round(spaces / total, 4)
    )

en spaces = 474476 space ratio = 0.161
tr spaces = 360375 space ratio = 0.1223
zh spaces = 45169 space ratio = 0.0153


## 20. Digit frequency

We count occurrences of the digits 0–9.

Numbers occur in Wikipedia text in dates, measurements, population counts, years, and other
contexts.

At the character level, digits are simply individual characters. A subword tokenizer such as BPE
may later learn recurring combinations involving digits.

In [35]:
#digits

digits = "0123456789"

for language in LANGUAGES:
    sentences = data["train"][language]
    text = "".join(sentences)

    digit_count = sum(text.count(d) for d in digits)

    print(language, "digit occurrences =", digit_count)

en digit occurrences = 56857
tr digit occurrences = 40098
zh digit occurrences = 150782


## 21. Characters unseen during training

A tokenizer should learn its vocabulary only from the training data.

We therefore check whether validation contains characters that never appeared in training.

If a validation character is absent from the training vocabulary, a character-level tokenizer
cannot assign it a normal character ID. In our implementation, such a character would be mapped
to the `<UNK>` token.

This experiment demonstrates why the `<UNK>` token is useful.

In [36]:
#unseen validation characters
for language in LANGUAGES:
    train_text = "".join(data["train"][language])
    valid_text = "".join(data["valid"][language])

    train_chars = set(train_text)
    valid_chars = set(valid_text)

    unseen = valid_chars - train_chars

    print(
        language,
        "validation characters unseen during training:",
        len(unseen)
    )

en validation characters unseen during training: 61
tr validation characters unseen during training: 26
zh validation characters unseen during training: 222


# Summary of Dataset Exploration

The dataset contains English, Turkish, and Mandarin Chinese Wikipedia text.

The three languages have approximately equal amounts of Unicode-character data, but they contain
different numbers of sentences because their average sentence lengths differ.

The character-level analysis showed that:
- each language has its own character distribution,
- some characters are shared across languages,
- other characters are language-specific,
- character frequencies differ substantially,
- whitespace behavior differs across languages,
- the dataset contains numbers and punctuation in addition to alphabetic/ideographic characters.

The character tokenizer uses the union of characters observed in the multilingual training data.
Each character is represented as one token, which means the number of character tokens is equal
to the number of Unicode characters in the input.

These observations provide the background for comparing character-level tokenization with BPE.
BPE can represent multiple characters as a single learned token when recurring character sequences
are frequent enough in the training corpus.

The important distinction is that BPE does not explicitly understand linguistic morphology or words.
Its vocabulary is learned from statistical patterns in the training data.

This dataset exploration therefore helps explain why English, Turkish, and Chinese may behave
differently under the same multilingual tokenizer.